In [ ]:
import numpy as np
import pandas as pd
import torch
import os
from joblib import Parallel, delayed, parallel_backend
from sklearn.preprocessing import StandardScaler
import sklearn.metrics


#-------------------------------------------Compute the HSIC value---------------
### fff compute the value of HSIC(alpha^T X,beta^T Y)
def fff(alpha, beta, data_X, data_Y, d1, d2, kernelx='d', kernely='d',r=0.7):
# Parameters:
#   alpha: projection vector for X
#   beta:  projection vector for Y
#   data_X: input matrix X (n x p)
#   data_Y: input matrix Y (n x q)
#   d1: degree/order for kernel on projected X
#   d2: degree/order for kernel on projected Y
#   kernelx: kernel type for X, 'np' (polynomial) or 'd' (L1 distance)
#   kernely: kernel type for Y, 'np' (polynomial) or 'd' (L1 distance)
#   r: weight parameters in statistical measures
    
    pro_X = data_X @ alpha
    pro_Y = data_Y @ beta
    n = len(pro_X)
    omega = np.array([1 + (-1) ** (i + 1) * r for i in range(data_X.shape[0])])

    if kernelx == 'np':
        # polynomial kernel K(pro_X,pro_X)=()
        DX = np.outer(pro_X, pro_X) ** d1
    elif kernelx == 'd':
        # L1 distance kernel
        DX = np.abs(pro_X.reshape(n, 1) - pro_X.reshape(1, n)) ** d1

    if kernely == 'np':
        # polynomial kernel
        DY = np.outer(pro_Y, pro_Y) ** d2
    elif kernely == 'd':
        # L1 distance kernel
        DY = np.abs(pro_Y.reshape(n, 1) - pro_Y.reshape(1, n)) ** d2

    A1 = (np.sum(DX * DY) / n / n +
          np.sum(DX / n / n) * np.sum(DY / n / n) -
          2 * np.mean((DX @ np.ones(n) / n) * (DY @ np.ones(n) / n) * omega))
    fn = -A1
    return fn

#-------------------------------------------Compute the gradient of HSIC---------------
def f(alpha, beta, data_X, data_Y, d1, d2, kernelx='d', kernely='d', mode='f',r=0.7):
    # Parameters:
    #   alpha, beta: projection vectors
    #   data_X, data_Y: input matrices (n x p) and (n x q)
    #   d1, d2: kernel degrees/orders
    #   kernelx, kernely: 'np' (polynomial) or 'd' (L1 distance)
    #   mode: 'f' (return function value),
    #         'pf' (return value + first derivative with respect to alpha),
    #         'LC' (return value + first derivative with respect to alpha + spectral norm of Hessian with respect to alpha)
    #   r: weight parameters in statistical measures
    
    # Convert to torch tensors with gradient tracking
    alpha = torch.tensor(alpha, requires_grad=True).to(torch.float64)
    beta = torch.tensor(beta, requires_grad=True).to(torch.float64)
    data_X = torch.tensor(data_X, requires_grad=True)
    data_Y = torch.tensor(data_Y, requires_grad=True)

    pro_X = data_X @ alpha
    pro_Y = data_Y @ beta
    n = len(pro_X)
    omega = torch.tensor(np.array([1 + (-1) ** (i + 1) * r for i in range(data_X.shape[0])]))

    if kernelx == 'np':
        # polynomial kernel
        DX = torch.outer(pro_X, pro_X) ** d1
    elif kernelx == 'd':
        # L1 distance kernel
        DX = torch.abs(pro_X.view(n, 1) - pro_X.view(1, n)) ** d1

    if kernely == 'np':
        # polynomial kernel
        DY = torch.outer(pro_Y, pro_Y) ** d2
    elif kernely == 'd':
        # L1 distance kernel
        DY = torch.abs(pro_Y.view(n, 1) - pro_Y.view(1, n)) ** d2

    A1 = (torch.mean(DX * DY) +
          torch.mean(DX) * torch.mean(DY) -
          2 * torch.mean((DX @ torch.ones(n).to(torch.float64) / n) *
                         (DY @ torch.ones(n).to(torch.float64) / n) * omega))
    fn = -A1

    if mode == 'f':
        return fn.detach().numpy()
    
    # First derivative w.r.t. alpha
    first_order = torch.autograd.grad(
        outputs=fn,
        inputs=alpha,
        grad_outputs=torch.ones_like(fn),
        create_graph=True,
        allow_unused=True
    )[0]

    if mode == 'pf':
        return fn.detach().numpy(), first_order.detach().numpy()

    # mode == 'LC' : compute Hessian spectral norm via randomized method
    jacobian = torch.zeros((len(first_order), len(alpha)))
    for i in range(len(first_order)):
        grad = torch.autograd.grad(
            outputs=first_order[i],
            inputs=alpha,
            grad_outputs=torch.ones_like(first_order[i]),
            create_graph=False,
            retain_graph=(i < len(first_order) - 1),
            allow_unused=True
        )[0]
        jacobian[i] = grad if grad is not None else torch.zeros_like(alpha)

    jacobian = (jacobian + jacobian.T) / 2
    LC = fast_spectral_norm_randomized(jacobian.numpy())#compute the matrix spectral norm

    return fn.detach().numpy(), first_order.detach().numpy(), LC




from sklearn.utils.extmath import randomized_svd

#Estimate the matrix spectral norm using the randomized SVD (singular value decomposition) method.
def fast_spectral_norm_randomized(A, k=1, n_oversamples=10, n_iter=2):
    # Estimate matrix 2-norm (largest singular value) using randomized SVD
    #   k: number of singular values to compute (default 1)
    #   n_oversamples: oversampling parameter, typical 10-20
    #   n_iter: power iteration steps, 2-5 gives high accuracy
    U, s, Vt = randomized_svd(
        A,
        n_components=k,
        n_oversamples=n_oversamples,
        n_iter=n_iter,
        random_state=42
    )
    return s[0]



#---------------------------------------Compute the variance--------------------

def ggg(alpha, beta, data_X, data_Y, d1, d2, kernelx='d', kernely='d',r=0.7):
    # Parameters:
    #   alpha, beta: projection vectors
    #   data_X, data_Y: input matrices
    #   d1, d2: kernel degrees
    #   kernelx, kernely: 'np' (polynomial) or 'd' (L1 distance)
    #   r: weight parameters in statistical measures
    pro_X = data_X @ alpha
    pro_Y = data_Y @ beta
    n = len(pro_X)

    if kernelx == 'np':
        # polynomial kernel
        DX = np.outer(pro_X, pro_X) ** d1
    elif kernelx == 'd':
        # L1 distance kernel
        DX = np.abs(pro_X.reshape(n, 1) - pro_X.reshape(1, n)) ** d1

    if kernely == 'np':
        # polynomial kernel
        DY = np.outer(pro_Y, pro_Y) ** d2
    elif kernely == 'd':
        # L1 distance kernel
        DY = np.abs(pro_Y.reshape(n, 1) - pro_Y.reshape(1, n)) ** d2

    gn = 4 * (r ** 2) * np.mean(((DX * DY) @ np.ones(n) / n - np.sum(DX * DY / n / n)) ** 2)
    return gn


#-------------------------------------------Compute the gradient of variance---------------
def g(alpha, beta, data_X, data_Y, d1, d2, kernelx='d', kernely='d', mode='f',r=0.7):
    # Parameters:
    #   alpha, beta: projection vectors
    #   data_X, data_Y: input matrices
    #   d1, d2: kernel degrees
    #   kernelx, kernely: 'np' (polynomial) or 'd' (L1 distance)
    #   mode: 'f' (return function value) or 'pf' (return value + first derivative)
    #   r: weight parameters in statistical measures

    # Convert to torch tensors with gradient tracking
    alpha = torch.tensor(alpha, requires_grad=True).to(torch.float64)
    beta = torch.tensor(beta, requires_grad=True).to(torch.float64)
    data_X = torch.tensor(data_X, requires_grad=True)
    data_Y = torch.tensor(data_Y, requires_grad=True)


    pro_X = data_X @ alpha
    pro_Y = data_Y @ beta
    n = len(pro_X)

    if kernelx == 'np':
        # polynomial kernel
        DX = torch.outer(pro_X, pro_X) ** d1
    elif kernelx == 'd':
        # L1 distance kernel
        DX = torch.abs(pro_X.view(n, 1) - pro_X.view(1, n)) ** d1

    if kernely == 'np':
        # polynomial kernel
        DY = torch.outer(pro_Y, pro_Y) ** d2
    elif kernely == 'd':
        # L1 distance kernel
        DY = torch.abs(pro_Y.view(n, 1) - pro_Y.view(1, n)) ** d2

    gn = 4 * (r ** 2) * torch.mean(((DX * DY) @ torch.ones(n).to(torch.float64) / n - torch.mean(DX * DY)) ** 2)

    if mode == 'f':
        return gn.detach().numpy()
    
    elif mode=='pf':
    # Compute first derivative w.r.t. alpha
        first_order = torch.autograd.grad(
            outputs=gn,
            inputs=alpha,
            grad_outputs=torch.ones_like(gn),
            create_graph=True,
            allow_unused=True
        )[0]

        return gn.detach().numpy(), first_order.detach().numpy()





#-----------------------------Compute the HSIC value for scalar X and Y---------------------------
def HSIC_scalar(data_X, data_Y, n):
    # Parameters:
    #   data_X, data_Y: input vectors (length n)
    #   n: sample size (should equal len(data_X))
    # Returns: HSIC estimate (scalar)

    distance_X = np.outer(data_X, data_X)
    distance_Y = np.outer(data_Y, data_Y)

    ones = np.ones(n)

    term1 = np.sum(distance_X * distance_Y)
    term2 = (ones @ distance_X @ ones) * (ones @ distance_Y @ ones) / ((n - 1) * (n - 2))
    term3 = 2 * (ones @ distance_X @ distance_Y @ ones) / (n - 2)

    atau = (term1 + term2 - term3) / (n * (n - 3))

    return atau




#----------For each component (each column), compute the centered kernel matrix--------------
def com_distance_list(data_X, d1, kernel):
    # Parameters:
    #   data_X: input matrix (n x p)
    #   d1: kernel degree/parameter
    #   kernel: 'Gauss', 'd' (L1 distance), or 'np' (polynomial)
    # Returns: list of centered kernel matrices (p x n x n)
    n = data_X.shape[0]
    p = data_X.shape[1]
    H = np.eye(n) - np.ones((n, n)) / n
    distance_Xlist = np.zeros((p, n, n))

    if kernel == 'Gauss':
        for i in range(p):
            aa = sklearn.metrics.pairwise_distances(np.reshape(data_X[:, i], [n, 1]))
            if np.median(aa) == 0:
                distance_Xlist[i] = np.exp(-aa ** 2) @ H
            else:
                distance_Xlist[i] = np.exp(-aa ** 2 / np.median(aa ** 2)) @ H
    elif kernel == 'd':
        for i in range(p):
            dist = sklearn.metrics.pairwise_distances(np.reshape(data_X[:, i], [n, 1]))
            distance_Xlist[i] = (dist ** d1) @ H
    elif kernel == 'np':
        for i in range(p):
            vec = np.reshape(data_X[:, i], [n, 1])
            distance_Xlist[i] = (np.outer(vec, vec) ** d1) @ H

    return distance_Xlist



def com_alpha_beta(distance_Xlist, distance_Ylist, n, p, q):
    # Parameters:
    #   distance_Xlist: list of centered kernel matrices for X (p x n x n)
    #   distance_Ylist: list for Y (q x n x n)
    #   n: sample size
    #   p, q: dimensions of X and Y
    # Returns: T matrix (p x q) of pairwise HSIC-like statistics

    row, col = np.triu_indices(n)
    w = np.where(row == col, 1.0, np.sqrt(2))
    w_tensor = torch.tensor(w, dtype=torch.float64)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    V_list = []
    for xi in distance_Xlist:
        xi_triu = torch.tensor(xi[row, col], dtype=torch.float64)
        xi_vec = xi_triu * w_tensor
        V_list.append(xi_vec)
    V = torch.stack(V_list).to(device)

    W_list = []
    for yi in distance_Ylist:
        yi_triu = torch.tensor(yi[row, col], dtype=torch.float64)
        yi_vec = yi_triu * w_tensor
        W_list.append(yi_vec)
    W = torch.stack(W_list).to(device)

    T = (V @ W.T).cpu().detach().numpy()
    return T

#---------------------------------Initial value---------------------------------
def first_value(data_X, data_Y, d1, d2, kernelx, kernely,r):
    # Parameters:
    #   data_X, data_Y: input matrices (n x p) and (n x q)
    #   d1, d2: kernel parameters
    #   kernelx, kernely: kernel types for X and Y
    # Returns: alpha (p-dim), beta (q-dim), g1 (scalar objective)

    p = data_X.shape[1]
    q = data_Y.shape[1]
    n = data_X.shape[0]

    distance_Xlist = com_distance_list(data_X, d1, kernelx)
    distance_Ylist = com_distance_list(data_Y, d2, kernely)

    alpha_beta_m = com_alpha_beta(distance_Xlist, distance_Ylist, n, p, q)

    alpha = np.max(alpha_beta_m, axis=1)
    beta = np.max(alpha_beta_m, axis=0)

    # normalize
    alpha = alpha / np.sqrt(alpha @ alpha)
    beta = beta / np.sqrt(beta @ beta)

    g1 = ggg(alpha, beta, data_X, data_Y, d1, d2, kernelx, kernely,r)

    return alpha, beta, g1



#------------------------Fix beta and compute the optimal projection direction alpha----------  
def iterate_scad(alpha, data_X, beta, data_Y, L, g1, la, pf, d1, d2, kernelx, kernely,
                 gamma=0, relaxation=0, abstol=1e-4, reltol=1e-3,r=0.7):
    # Parameters:
    #   alpha, beta: projection vectors
    #   data_X, data_Y: input matrices
    #   L: Lipschitz constant estimate
    #   g1: reference value
    #   la: regularization parameter
    #   pf: previous function value?
    #   d1,d2,kernelx,kernely: kernel parameters
    # Returns: updated alpha (numpy array)

    alpha2 = np.copy(alpha)
    u1 = np.zeros(len(alpha))
    u2 = 0.0
    p = data_X.shape[1]

    g_value, pg = g(alpha, beta, data_X, data_Y, d1, d2, kernelx, kernely, mode='pf',r=r)
    Rho = 0.05
    R = 0.05

    pgT = np.reshape(pg, (len(pg), 1))
    pgTT = pgT @ pgT.T
    pg_alpha = pg @ alpha
    pg_pg = pg @ pg

    # Convert to torch tensors
    devices = [torch.tensor(x) for x in [pf, L, alpha, alpha2, R, u1, u2, g1, g_value, pg, pg_alpha, pgTT, la]]
    pf, L, alpha, alpha2, R, u1, u2, g1, g_value, pg, pg_alpha, pgTT, la = devices

    p = len(alpha)
    for j in range(200):
        # Compute intermediate update
        alpha1c = -pf + L * alpha + R * alpha2 - R * u1 - R * (g_value - g1 + u2) * pg + R * pg_alpha * pg

        eye_term = torch.eye(p, device='cpu') / (L + R)
        A = eye_term - (L + R) ** (-2) * R * pgTT / (1 + R / (L + R) * pg_pg)
        alpha1c = A @ alpha1c

        alpha1c = R * (alpha1c + u1)

        # Soft-thresholding (SCAD-like)
        a1 = (alpha1c - la + torch.abs(alpha1c - la)) / 2
        a2 = (alpha1c + la - torch.abs(alpha1c + la)) / 2
        alpha2c = (a1 + a2) / R

        # Project onto unit ball
        alpha2_norm = torch.norm(alpha2c)
        if alpha2_norm > 1:
            alpha2c /= alpha2_norm

        # Update dual variables
        u1 = u1 + Rho * (alpha1c - alpha2c)
        u2 = u2 + Rho * (g_value + pg @ (alpha1c - alpha) - g1) / g1

        # Check convergence
        dual_res = torch.norm(alpha2c - alpha2)

        alpha2 = alpha2c
        if j > 0 and dual_res < np.sqrt(p) * 1e-2:
            break

    newalpha = alpha2
    return newalpha.numpy()

#---------------------Use ADMM to alternately compute the projection directions-----------------
def com_direction_cv(alpha, beta, g1, data_X, data_Y, lax, lay, d1, d2, kernelx, kernely,r):
    # Parameters:
    #   alpha, beta: current projection vectors
    #   g1: reference value
    #   data_X, data_Y: input matrices
    #   lax, lay: regularization parameters
    #   d1,d2,kernelx,kernely: kernel parameters
    # Returns: updated alpha, beta

    i = 0
    while True:
        if alpha @ alpha == 0 or beta @ beta == 0:
            i += 1
            L = 1
        else:
            i += 1
            f_value, pf, L = f(alpha, beta, data_X, data_Y, d1, d2, kernelx, kernely, 'LC',r)
            L = max(L, 1)

        alpha1c = iterate_scad(alpha, data_X, beta, data_Y, L, g1, lax, pf, d1, d2, kernelx, kernely,r=r)

        if alpha1c @ alpha1c == 0:
            L = 1
        else:
            f_value, pf, L = f(beta, alpha1c, data_Y, data_X, d2, d1, kernely, kernelx, 'LC',r)
            L = max(L, 1)

        beta1c = iterate_scad(beta, data_Y, alpha1c, data_X, L, g1, lay, pf, d2, d1, kernely, kernelx,r=r)

        alpha = alpha1c
        beta = beta1c

        # Stopping criterion
        term1 = abs(fff(alpha1c, beta1c, data_X, data_Y, d1, d2, kernelx, kernely,r))
        term2 = np.sqrt(ggg(alpha, beta, data_X, data_Y, d1, d2, kernelx, kernely,r))
        term3 = np.sqrt(ggg(alpha1c, beta1c, data_X, data_Y, d1, d2, kernelx, kernely,r))
        term4 = abs(fff(alpha, beta, data_X, data_Y, d1, d2, kernelx, kernely,r))

        if i > 200 or term1 * term2 <= 1.1 * term3 * term4:
            break

    return alpha, beta

#------------------------------Select the penalty parameter--------------------------------
def slection_lambda(la_candidates,alpha0, beta0, g1, data_X, data_Y, d1, d2, kernelx, kernely,r):
    # Parameters:
    #   alpha0, beta0: initial projection vectors
    #   g1: reference value
    #   data_X, data_Y: input matrices
    #   d1,d2,kernelx,kernely: kernel parameters
    # Returns: selected alpha, beta for best lambda pair

    # Candidate lambda values (grid)
    la_candidates = [(i, j) for i in la_candidates for j in la_candidates]

    # Storage for results
    cv_scores = {la: [] for la in la_candidates}
    alpha_la = {la: [] for la in la_candidates}
    beta_la = {la: [] for la in la_candidates}

    n_jobs = min(8, os.cpu_count())

    with parallel_backend("loky"):
        results = Parallel(n_jobs=n_jobs)(
            delayed(com_direction_cv)(alpha0, beta0, g1, data_X, data_Y, la[0], la[1], d1, d2, kernelx, kernely,r)
            for la in la_candidates
        )

    for i, la in enumerate(la_candidates):
        alpha, beta = results[i]

        # Threshold small coefficients
        alpha = alpha * np.float64(np.abs(alpha) > 1e-3)
        beta = beta * np.float64(np.abs(beta) > 1e-3)

        alpha_la[la] = alpha
        beta_la[la] = beta

        cv_scores[la].append(com_p_value(alpha, beta, data_X, data_Y, d1, d2, kernelx, kernely,r))

    # Select best lambda pair minimizing sum of scores
    avg_scores = {la: np.sum(scores) for la, scores in cv_scores.items()}
    best_la = min(avg_scores, key=avg_scores.get)

    return alpha_la[best_la], beta_la[best_la]


#---------------------------Evaluate the penalty parameter-------------------------------------
def com_p_value(alpha, beta, data_X, data_Y, d1, d2, kernelx, kernely,r):
    # Parameters:
    #   alpha, beta: projection vectors
    #   data_X, data_Y: input matrices
    #   d1,d2,kernelx,kernely: kernel parameters
    # Returns: p-value (1 if either vector is zero)

    if alpha @ alpha == 0 or beta @ beta == 0:
        return 1

    atau = -fff(alpha, beta, data_X, data_Y, d1, d2, kernelx, kernely,r) / np.sqrt(
        ggg(alpha, beta, data_X, data_Y, d1, d2, kernelx, kernely,r)
    )

    p1_value = []
    for _ in range(1000):
        # Permute Y
        permute_index = np.random.permutation(data_Y.shape[0])
        data_Y_perm = data_Y[permute_index, :]

        atau1 = -fff(alpha, beta, data_X, data_Y_perm, d1, d2, kernelx, kernely,r) / np.sqrt(
            ggg(alpha, beta, data_X, data_Y_perm, d1, d2, kernelx, kernely,r)
        )
        p1_value.append(atau1)

    return np.mean(np.array(p1_value) > atau)


def com_significance(atau, atau_bt, significance_level):
    # Test if observed statistic exceeds bootstrap quantile
    #   atau: observed statistic
    #   atau_bt: bootstrap/null distribution
    #   significance_level: e.g., 0.05
    # Returns: 1 if significant, else 0
    if atau > np.percentile(atau_bt, 100 - significance_level * 100):
        return 1
    else:
        return 0




In [ ]:

#-------------------------------Our Proposed Method------------------------------------------------------
def com_PIKE(data_X, data_Y, kernelx, kernely, significance_level, r, la_candidates, ratio=0.5):
    # Parameters:
    #   data_X, data_Y: input matrices (n x p) and (n x q)
    #   significance_level: significance level for hypothesis test
    #   r: weight parameter
    #   kernelx, kernely: kernel types for X and Y, 'np' (polynomial) or 'd' (L1 distance)
    #   la_candidates: list of candidate lambda pairs
    #   ratio: proportion of data used for training (default 0.5)
    # Returns: binary decisions for three tests (t1 for d1=1, t2 for d1=2, t3 for PIKE)

    n = data_X.shape[0]
    n_train = int(n * ratio)
    n_test = n - n_train

    data_Xtrain = data_X[:n_train, :]
    data_Xtest = data_X[n_train:, :]
    data_Ytrain = data_Y[:n_train, :]
    data_Ytest = data_Y[n_train:, :]

    scaler = StandardScaler()
    data_Xtrain = scaler.fit_transform(data_Xtrain)
    data_Ytrain = scaler.fit_transform(data_Ytrain)
    data_Xtest = scaler.fit_transform(data_Xtest)
    data_Ytest = scaler.fit_transform(data_Ytest)

    # Storage for loadings and statistics
    a = {'1': {'1': []}, '2': {'1': [], '2': []}}
    b = {'1': {'1': []}, '2': {'1': [], '2': []}}
    c = []
    c_weight = []



    for d1 in [1, 2]:
        for d2 in [1]:
            alpha0, beta0, g1 = first_value(data_Xtrain, data_Ytrain, d1, d2, kernelx, kernely, r)
            alpha, beta = slection_lambda(la_candidates, alpha0, beta0, 1, data_Xtrain, data_Ytrain, d1, d2, kernelx, kernely, r)

            if alpha @ alpha == 0 or beta @ beta == 0:
                atau = 0
                atau_weight = 0
            else:
                sqrt_n_test = np.sqrt(n_test)
                sqrt_n_train = np.sqrt(n_train)
                atau = -fff(alpha, beta, data_Xtest, data_Ytest, d1, d2, kernelx, kernely, r) / np.sqrt(
                    ggg(alpha, beta, data_Xtest, data_Ytest, d1, d2, kernelx, kernely, r)) * sqrt_n_test
                atau_weight = -fff(alpha, beta, data_Xtrain, data_Ytrain, d1, d2, kernelx, kernely, r) / np.sqrt(
                    ggg(alpha, beta, data_Xtrain, data_Ytrain, d1, d2, kernelx, kernely, r)) * sqrt_n_train

            a[str(d1)][str(d2)].append(alpha)
            b[str(d1)][str(d2)].append(beta)
            c.append(atau)
            c_weight.append(atau_weight)

    # Compute weights based on training set statistics
    c_weight_arr = np.array(c_weight)
    weight = np.exp(c_weight_arr) / np.sum(np.exp(c_weight_arr))  # weights for PIKE

    alpha1 = a['1']['1'][0]
    beta1 = b['1']['1'][0]
    alpha2 = a['2']['1'][0]
    beta2 = b['2']['1'][0]

    # Kernels on test set
    pro_X1 = data_Xtest @ alpha1
    pro_Y1 = data_Ytest @ beta1
    DX1 = np.outer(pro_X1, pro_X1)
    DY1 = np.outer(pro_Y1, pro_Y1)

    pro_X2 = data_Xtest @ alpha2
    pro_Y2 = data_Ytest @ beta2
    DX2 = np.outer(pro_X2, pro_X2) ** 2
    DY2 = np.outer(pro_Y2, pro_Y2)

    # Correlation matrix for the two component statistics
    corr = np.cov(np.mean(DX1 * DY1, axis=1), np.mean(DX2 * DY2, axis=1))
    std1 = np.std(np.mean(DX1 * DY1, axis=1))
    std2 = np.std(np.mean(DX2 * DY2, axis=1))
    if std1 > 0:
        corr[0, 0] = corr[0, 0] / np.var(np.mean(DX1 * DY1, axis=1))
        corr[0, 1] = corr[0, 1] / std1
        corr[1, 0] = corr[1, 0] / std1
    if std2 > 0:
        corr[1, 1] = corr[1, 1] / np.var(np.mean(DX2 * DY2, axis=1))
        corr[0, 1] = corr[0, 1] / std2
        corr[1, 0] = corr[1, 0] / std2

    c_arr = np.array(c)
    PIKE_statistic = np.sum(weight * c_arr) / np.sqrt(
        np.sum(weight ** 2 * np.diag(corr)) + 2 * weight[0] * weight[1] * corr[0, 1]
    )  # test statistic

    # Permutation test
    data_Y_perm = np.copy(data_Ytest)
    p1_value = []   # for d1=1 component
    p2_value = []   # for d1=2 component
    p3_value = []   # for PIKE statistic

    for _ in range(1000):
        np.random.shuffle(data_Y_perm)

        c1 = []
        for d1 in [1, 2]:
            for d2 in [1]:
                alpha = a[str(d1)][str(d2)][0]
                beta = b[str(d1)][str(d2)][0]

                if alpha @ alpha == 0 or beta @ beta == 0:
                    atau1 = 0
                else:
                    atau1 = -fff(alpha, beta, data_Xtest, data_Y_perm, d1, d2, kernelx, kernely, r) / np.sqrt(
                        ggg(alpha, beta, data_Xtest, data_Y_perm, d1, d2, kernelx, kernely, r)) * sqrt_n_test
                c1.append(atau1)
                if d1 == 1:
                    p1_value.append(atau1)
                else:
                    p2_value.append(atau1)

        # Recompute correlation for the permuted Y
        pro_Y1_perm = data_Y_perm @ beta1
        DY1_perm = np.outer(pro_Y1_perm, pro_Y1_perm)
        pro_Y2_perm = data_Y_perm @ beta2
        DY2_perm = np.outer(pro_Y2_perm, pro_Y2_perm)

        corr_perm = np.cov(np.mean(DX1 * DY1_perm, axis=1), np.mean(DX2 * DY2_perm, axis=1))
        std1_perm = np.std(np.mean(DX1 * DY1_perm, axis=1))
        std2_perm = np.std(np.mean(DX2 * DY2_perm, axis=1))
        if std1_perm > 0:
            corr_perm[0, 0] = corr_perm[0, 0] / np.var(np.mean(DX1 * DY1_perm, axis=1))
            corr_perm[0, 1] = corr_perm[0, 1] / std1_perm
            corr_perm[1, 0] = corr_perm[1, 0] / std1_perm
        if std2_perm > 0:
            corr_perm[1, 1] = corr_perm[1, 1] / np.var(np.mean(DX2 * DY2_perm, axis=1))
            corr_perm[0, 1] = corr_perm[0, 1] / std2_perm
            corr_perm[1, 0] = corr_perm[1, 0] / std2_perm

        c1_arr = np.array(c1)
        atau1_fusenew = np.sum(weight * c1_arr) / np.sqrt(
            np.sum(weight ** 2 * np.diag(corr_perm)) + 2 * weight[0] * weight[1] * corr_perm[0, 1]
        )
        p3_value.append(atau1_fusenew)

    t1 = com_significance(c[0], p1_value, significance_level)
    t2 = com_significance(c[1], p2_value, significance_level)
    t3 = com_significance(PIKE_statistic, p3_value, significance_level)

    return t1, t2, t3
